In [4]:
# ============================================================
# RGCov-VAR(2): Eight Financial Time Series
# Reconstruction for IJCEE Manuscript
# ============================================================

!pip -q install yfinance arch jax scipy pandas numpy

import numpy as np
import pandas as pd
import yfinance as yf
import jax
import jax.numpy as jnp

from arch import arch_model
from scipy.optimize import minimize

jax.config.update("jax_enable_x64", True)

# ------------------------------------------------------------
# 1. SETTINGS
# ------------------------------------------------------------

H = 5
ridge_lambda = 0.001

N = 8
p = 2
n_starts = 5

gradient_tolerance = 1e-6
random_seed = 42

np.random.seed(random_seed)

print("RGCov reconstruction")
print("H =", H)
print("Lambda =", ridge_lambda)

# ------------------------------------------------------------
# 2. DOWNLOAD FINANCIAL DATA
# ------------------------------------------------------------

tickers = {
    "SP500": "^GSPC",
    "NASDAQ": "^IXIC",
    "DOW": "^DJI",
    "VIX": "^VIX",
    "GOLD": "GC=F",
    "OIL": "CL=F",
    "TREASURY10Y": "^TNX",
    "DOLLAR": "DX-Y.NYB"
}

series = {}

for name, ticker in tickers.items():

    print("Downloading:", name)

    df = yf.download(
        ticker,
        start="2010-01-01",
        end="2024-01-01",
        auto_adjust=True,
        progress=False
    )

    if df.empty:
        raise ValueError(f"No data downloaded for {ticker}")

    close = df["Close"]

    if isinstance(close, pd.DataFrame):
        close = close.iloc[:, 0]

    series[name] = close

prices = pd.concat(series, axis=1)

prices = prices.dropna()

prices = prices[(prices > 0).all(axis=1)]

print("\nPrice data shape:", prices.shape)

# ------------------------------------------------------------
# 3. DAILY LOG CHANGES
# ------------------------------------------------------------

returns = np.log(prices).diff().dropna()

returns = (
    returns - returns.mean()
) / returns.std(ddof=0)

print("Return data shape:", returns.shape)

# ------------------------------------------------------------
# 4. GARCH(1,1) FILTERING
# ------------------------------------------------------------

filtered = pd.DataFrame(index=returns.index)

for column in returns.columns:

    print("GARCH filtering:", column)

    y = returns[column].dropna()

    model = arch_model(
        y,
        mean="Constant",
        vol="GARCH",
        p=1,
        q=1,
        dist="normal",
        rescale=False
    )

    result = model.fit(disp="off")

    filtered[column] = result.std_resid

filtered = filtered.replace(
    [np.inf, -np.inf], np.nan
).dropna()

Y = filtered.to_numpy(dtype=np.float64)

T, N = Y.shape

print("\nFiltered observations:", T)
print("Number of variables:", N)

# ------------------------------------------------------------
# 5. CREATE VAR(2) DATA
# ------------------------------------------------------------

def create_var_data(Y, p):

    T, N = Y.shape

    target = Y[p:]

    X = np.concatenate(
        [Y[p-lag:T-lag] for lag in range(1, p+1)],
        axis=1
    )

    return X, target

X, target = create_var_data(Y, p)

X_jax = jnp.asarray(X)
target_jax = jnp.asarray(target)

number_parameters = N * N * p

print("Number of VAR parameters:", number_parameters)

# ------------------------------------------------------------
# 6. OLS INITIALIZATION
# ------------------------------------------------------------

B_ols = np.linalg.lstsq(
    X, target, rcond=None
)[0]

beta_initial = B_ols.reshape(-1)

# ------------------------------------------------------------
# 7. RGCov OBJECTIVE
# ------------------------------------------------------------

# g(u) = (u, u^2)
# We calculate lagged covariance matrices separately for
# residual levels and squared residuals.
#
# Objective:
# sum_h ||Gamma_h(level)||_F^2
# + sum_h ||Gamma_h(squared)||_F^2
# + lambda * ||beta||_2^2

def rgcov_objective(beta):

    B = beta.reshape((N*p, N))

    residuals = target_jax - X_jax @ B

    residuals = residuals - jnp.mean(
        residuals, axis=0
    )

    squared = residuals**2

    squared = squared - jnp.mean(
        squared, axis=0
    )

    loss = 0.0

    for h in range(1, H+1):

        covariance_levels = (
            residuals[h:].T @ residuals[:-h]
        ) / (residuals.shape[0] - h)

        covariance_squared = (
            squared[h:].T @ squared[:-h]
        ) / (squared.shape[0] - h)

        loss += jnp.sum(covariance_levels**2)

        loss += jnp.sum(covariance_squared**2)

    penalty = ridge_lambda * jnp.sum(beta**2)

    return loss + penalty

objective_jit = jax.jit(rgcov_objective)

gradient_jit = jax.jit(
    jax.grad(rgcov_objective)
)

def objective_numpy(beta):
    return float(objective_jit(jnp.asarray(beta)))

def gradient_numpy(beta):
    return np.asarray(
        gradient_jit(jnp.asarray(beta)),
        dtype=np.float64
    )

# ------------------------------------------------------------
# 8. MULTI-START L-BFGS-B OPTIMIZATION
# ------------------------------------------------------------

bounds = [(-5, 5)] * number_parameters

best_result = None
best_certified_result = None

for start in range(n_starts):

    print(f"\nOptimization start {start+1}/{n_starts}")

    if start == 0:
        beta_start = beta_initial.copy()
    else:
        beta_start = (
            beta_initial
            + np.random.normal(
                0, 0.01,
                size=number_parameters
            )
        )

    beta_start = np.clip(beta_start, -5, 5)

    result = minimize(
        objective_numpy,
        beta_start,
        method="L-BFGS-B",
        jac=gradient_numpy,
        bounds=bounds,
        options={
            "maxiter": 3000,
            "ftol": 1e-14,
            "gtol": 1e-10,
            "maxls": 50
        }
    )

    grad_norm = np.max(
        np.abs(gradient_numpy(result.x))
    )

    certified = (
        result.success
        and np.isfinite(result.fun)
        and grad_norm < gradient_tolerance
    )

    print("Objective:", result.fun)
    print("Gradient norm:", grad_norm)
    print("Optimizer success:", result.success)
    print("Certified:", certified)

    if (
        best_result is None
        or result.fun < best_result.fun
    ):
        best_result = result

    if certified:
        if (
            best_certified_result is None
            or result.fun < best_certified_result.fun
        ):
            best_certified_result = result

# ------------------------------------------------------------
# 9. SELECT RESULT
# ------------------------------------------------------------

if best_certified_result is not None:
    final_result = best_certified_result
else:
    final_result = best_result
    print(
        "\nWARNING: No start met the certification rule."
    )

beta_hat = final_result.x

final_gradient = np.max(
    np.abs(gradient_numpy(beta_hat))
)

certified = (
    final_result.success
    and np.isfinite(final_result.fun)
    and final_gradient < gradient_tolerance
)

# ------------------------------------------------------------
# 10. COMPANION MATRIX AND ROOT CLASSIFICATION
# ------------------------------------------------------------

B = beta_hat.reshape((N*p, N))

# B has shape (Np, N), with lag matrices stacked vertically.
# The VAR companion matrix uses their transposes.

A1 = B[:N, :].T
A2 = B[N:2*N, :].T

companion = np.zeros((N*p, N*p))

companion[:N, :N] = A1
companion[:N, N:2*N] = A2

companion[N:2*N, :N] = np.eye(N)

eigenvalues = np.linalg.eigvals(companion)

root_moduli = np.abs(eigenvalues)

spectral_radius = np.max(root_moduli)

causal_roots = np.sum(root_moduli < 1)
noncausal_roots = np.sum(root_moduli > 1)

unit_roots = np.sum(
    np.isclose(root_moduli, 1, atol=1e-8)
)

# ------------------------------------------------------------
# 11. FINAL RESULTS
# ------------------------------------------------------------

results = {
    "H": H,
    "ridge_lambda": ridge_lambda,
    "N": N,
    "VAR_lags": p,
    "observations": T,
    "parameters": number_parameters,
    "objective": float(final_result.fun),
    "gradient_infinity_norm": float(final_gradient),
    "optimizer_success": bool(final_result.success),
    "certified": bool(certified),
    "spectral_radius": float(spectral_radius),
    "causal_roots": int(causal_roots),
    "noncausal_roots": int(noncausal_roots),
    "unit_roots": int(unit_roots)
}

print("\n" + "="*55)
print("RECONSTRUCTED RESULTS")
print("="*55)

for key, value in results.items():
    print(f"{key}: {value}")

# ------------------------------------------------------------
# 12. SAVE RESULTS
# ------------------------------------------------------------

pd.DataFrame(
    [results]
).to_csv(
    "rgcov_n8_results.csv",
    index=False
)

pd.DataFrame(
    {
        "real": eigenvalues.real,
        "imaginary": eigenvalues.imag,
        "modulus": root_moduli
    }
).to_csv(
    "rgcov_n8_eigenvalues.csv",
    index=False
)

print("\nResults saved successfully.")

RGCov reconstruction
H = 5
Lambda = 0.001
Downloading: SP500
Downloading: NASDAQ
Downloading: DOW
Downloading: VIX
Downloading: GOLD
Downloading: OIL
Downloading: TREASURY10Y
Downloading: DOLLAR

Price data shape: (3517, 8)
Return data shape: (3516, 8)
GARCH filtering: SP500
GARCH filtering: NASDAQ
GARCH filtering: DOW
GARCH filtering: VIX
GARCH filtering: GOLD
GARCH filtering: OIL
GARCH filtering: TREASURY10Y
GARCH filtering: DOLLAR

Filtered observations: 3516
Number of variables: 8
Number of VAR parameters: 128

Optimization start 1/5
Objective: 2.3822631582911953
Gradient norm: 7.802449872664822e-07
Optimizer success: True
Certified: True

Optimization start 2/5
Objective: 2.382263158291926
Gradient norm: 1.462417873125233e-06
Optimizer success: True
Certified: False

Optimization start 3/5
Objective: 2.3822631582914178
Gradient norm: 8.60571726572019e-07
Optimizer success: True
Certified: True

Optimization start 4/5
Objective: 2.3822631582912726
Gradient norm: 6.469458041321182e-